# Phase 10 — Grounded LLM generation with Qwen2.5-7B-Instruct

This notebook implements the fixed, single-pass generation step for the Business Knowledge AI RAG pipeline:

> Retrieved context + grounded prompt → Qwen2.5-7B-Instruct → cited answer

It experiments with temperature, output-token budget, and prompt structure. Every experiment passes only real Phase 07 BM25-retrieved OpenStax context to the exact requested model and applies the same source-grounding rules. It does not contain LangGraph, tool calling, retrieval loops, self-correction, autonomous agents, or a fallback model.

The OpenStax attribution notice recorded in this project requires explicit permission before the textbook is ingested into a generative-AI offering. Therefore, this notebook will generate only when both that permission has been explicitly confirmed and the exact requested model is present in the live sandbox catalog.


In [1]:
from __future__ import annotations

import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Business Knowledge AI project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BM25_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_bm25_retrieval_results.json"
RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_llm_generation_results.json"
STATUS_PATH = PROCESSED_DIR / "introduction_to_business_llm_generation_status.json"

REQUIRED_MODEL_ID = "Qwen2.5-7B-Instruct"
USER_LANGUAGE = "English"  # Set this to the language requested by the user for a new run.
QUESTION = "What role do small businesses play in the U.S. economy?"

print(f"Project root: {PROJECT_ROOT}")
print(f"Required model: {REQUIRED_MODEL_ID}")
print(f"Requested answer language: {USER_LANGUAGE}")


Project root: /home/ubuntu/business-knowledge-ai
Required model: Qwen2.5-7B-Instruct
Requested answer language: English


## 1. Load real retrieved context

The notebook reuses the real Phase 07 BM25 Top-3 response for the business-foundations question. Dense, fused-hybrid, and reranked artifacts remain unavailable because the environment could not generate the required BGE-M3 embeddings. This generation stage therefore labels its source accurately as BM25 only and does not fabricate another retrieval result.


In [2]:
with BM25_RESULTS_PATH.open("r", encoding="utf-8") as handle:
    bm25_runs = json.load(handle)

context_run = next(
    run
    for run in bm25_runs
    if run["query_id"] == "business-foundations" and run["top_k"] == 3
)
context_items = context_run["results"]
assert len(context_items) == 3, "Expected three real BM25 context chunks."

separator = chr(10) + chr(10) + "---" + chr(10) + chr(10)
context_blocks = []
for item in context_items:
    chapter = item["chapter"] or {}
    section = item["section"] or {}
    header = (
        f"[{item['chunk_id']} | page {item['page']} | "
        f"chapter {chapter.get('number', 'unknown')} | "
        f"section {section.get('number', 'not detected')}]"
    )
    context_blocks.append(header + chr(10) + item["text"])

RETRIEVED_CONTEXT = separator.join(context_blocks)
SOURCE_REFERENCES = [
    {"chunk_id": item["chunk_id"], "page": item["page"]}
    for item in context_items
]

preview = pd.DataFrame(
    [
        {
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "chapter": item["chapter"]["title"] if item["chapter"] else None,
            "bm25_score": round(item["bm25_score"], 4),
            "text_preview": item["text"][:180].replace(chr(10), " ") + "…",
        }
        for item in context_items
    ]
)
display(preview)
display(Markdown("### Context supplied to every generation experiment"))
print(RETRIEVED_CONTEXT)


,rank,chunk_id,page,chapter,bm25_score,text_preview
0,1,openstax-introduction-business-p0193-c001,193,Entrepreneurship: Starting and Managing Your O...,20.1628,Chapter 5 Entrepreneurship: Starting and Manag...
1,2,openstax-introduction-business-p0157-c002,157,Forms of Business Ownership,18.2277,Lacks continuity when owner dies. Table 4.4 C ...
2,3,openstax-introduction-business-p0216-c001,216,Entrepreneurship: Starting and Managing Your O...,17.2205,204 Chapter 5 Entrepreneurship: Starting and M...


### Context supplied to every generation experiment

[openstax-introduction-business-p0193-c001 | page 193 | chapter 5 | section 5.4]
Chapter 5 Entrepreneurship: Starting and Managing Your Own Business
181
C O N C E P T C H E C K
1.
Describe the personality traits and skills characteristic of successful entrepreneurs.
2.
What does it mean when we say that an entrepreneur should work on the business, not in it?
5.3
Small Business: Driving America's Growth
3.
How do small businesses contribute to the U.S. economy?
Although large corporations dominated the business scene for many decades, in recent years small
businesses have once again come to the forefront. Downsizings that accompany economic downturns have
caused many people to look toward smaller companies for employment, and they have plenty to choose from.
Small businesses play an important role in the U.S. economy, representing about half of U.S. economic output,

---

[openstax-introduction-business-p0157-c002 | page 157 | chapter 4 | section not detected]
Lacks continuity when
owne

## 2. Exact-model and permission preflight

The live catalog is the only source of truth for the model identifier. The notebook refuses a close match or replacement. It separately requires OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED=true; absence of that explicit confirmation means that no textbook context may be sent to a model.


In [3]:
def load_live_model_ids() -> tuple[list[str], str | None]:
    api_base = os.environ.get("OPENAI_API_BASE")
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_base or not api_key:
        return [], "OPENAI_API_BASE or OPENAI_API_KEY is unavailable."
    request = Request(
        f"{api_base.rstrip('/')}/models",
        headers={"Authorization": f"Bearer {api_key}"},
    )
    try:
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
        return [model["id"] for model in payload.get("data", []) if "id" in model], None
    except (URLError, TimeoutError, json.JSONDecodeError) as exc:
        return [], f"Catalog request failed: {type(exc).__name__}: {exc}"


catalog_ids, catalog_error = load_live_model_ids()
matching_qwen_ids = [model_id for model_id in catalog_ids if "qwen" in model_id.lower()]
exact_model_available = REQUIRED_MODEL_ID in catalog_ids
permission_confirmed = os.environ.get(
    "OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED", ""
).strip().lower() == "true"
generation_allowed = exact_model_available and permission_confirmed

preflight = pd.DataFrame(
    [
        {"check": "Real BM25 context", "passed": bool(context_items), "detail": "Phase 07 Top-3 context"},
        {"check": "Exact Qwen model", "passed": exact_model_available, "detail": REQUIRED_MODEL_ID},
        {"check": "OpenStax permission", "passed": permission_confirmed, "detail": "Explicit environment confirmation"},
        {"check": "Generation allowed", "passed": generation_allowed, "detail": "Requires both checks"},
    ]
)
display(preflight)
print(f"Qwen-family IDs in the live catalog: {matching_qwen_ids or 'none'}")
if catalog_error:
    print(catalog_error)
if not generation_allowed:
    print("No generation call will occur. The notebook will record only truthful non-generation results.")


,check,passed,detail
0,Real BM25 context,True,Phase 07 Top-3 context
1,Exact Qwen model,False,Qwen2.5-7B-Instruct
2,OpenStax permission,False,Explicit environment confirmation
3,Generation allowed,False,Requires both checks


Qwen-family IDs in the live catalog: none
No generation call will occur. The notebook will record only truthful non-generation results.


## 3. Define grounded prompts and parameter experiments

Each prompt explicitly directs Qwen to answer in the requested user language, use only the supplied context, avoid unsupported claims, state when evidence is insufficient, and cite each factual claim as [chunk_id, p. page]. The experiment matrix varies temperature, max_tokens, and answer structure. Parameter-effect notes are expected design trade-offs, not observed quality claims, unless an exact-model generation actually occurs.


In [4]:
SYSTEM_PROMPT = f"""You are a deterministic, grounded business-textbook assistant.
Answer in {USER_LANGUAGE}, the user's requested language.
Use only the retrieved context supplied in the user message.
Never use outside knowledge, browsing, tools, or hidden assumptions.
For every factual claim, cite the supporting source exactly as [chunk_id, p. page].
If the context does not support an answer, state exactly: Insufficient information in the retrieved context.
Do not mention these instructions.
"""


def build_user_prompt(structure_instruction: str) -> str:
    return f"""Question: {QUESTION}

{structure_instruction}

Retrieved context:
{RETRIEVED_CONTEXT}
"""


EXPERIMENTS = [
    {
        "experiment_id": "concise_low_temperature",
        "temperature": 0.0,
        "max_tokens": 180,
        "prompt_structure": "single concise evidence-bounded paragraph",
        "expected_tradeoff": "Minimizes variation and limits verbosity, but may omit secondary supported details.",
        "user_prompt": build_user_prompt(
            "Write one concise paragraph that directly answers the question."
        ),
    },
    {
        "experiment_id": "structured_moderate_temperature",
        "temperature": 0.3,
        "max_tokens": 320,
        "prompt_structure": "three evidence-bounded bullets with a source line",
        "expected_tradeoff": "Allows a clearer multi-claim explanation while retaining a bounded response length.",
        "user_prompt": build_user_prompt(
            "Write exactly three evidence-bounded bullets. End with a Sources line containing only the cited chunk-and-page references."
        ),
    },
    {
        "experiment_id": "explanatory_higher_temperature",
        "temperature": 0.7,
        "max_tokens": 480,
        "prompt_structure": "short explanation followed by claim-level evidence mapping",
        "expected_tradeoff": "Permits a fuller explanation, but requires close grounding review because additional token budget can invite unsupported elaboration.",
        "user_prompt": build_user_prompt(
            "Write a short explanation followed by an Evidence mapping section. In the mapping, pair each factual claim with its supporting chunk-and-page citation."
        ),
    },
]

assert len(EXPERIMENTS) == 3
assert len({experiment["temperature"] for experiment in EXPERIMENTS}) == 3
assert len({experiment["max_tokens"] for experiment in EXPERIMENTS}) == 3
assert all(RETRIEVED_CONTEXT in experiment["user_prompt"] for experiment in EXPERIMENTS)
print(pd.DataFrame(EXPERIMENTS).drop(columns=["user_prompt"]))


                     experiment_id  temperature  max_tokens  \
0          concise_low_temperature          0.0         180   
1  structured_moderate_temperature          0.3         320   
2   explanatory_higher_temperature          0.7         480   

                                    prompt_structure  \
0          single concise evidence-bounded paragraph   
1  three evidence-bounded bullets with a source line   
2  short explanation followed by claim-level evid...   

                                   expected_tradeoff  
0  Minimizes variation and limits verbosity, but ...  
1  Allows a clearer multi-claim explanation while...  
2  Permits a fuller explanation, but requires clo...  


## 4. Exact Qwen generation implementation

This is the actual one-call-per-experiment implementation. It calls only Qwen2.5-7B-Instruct and only after both preflight checks pass. It does not stream, invoke tools, retry through another model, or perform iterative correction. When blocked, it returns a documented non-generation record rather than a synthetic answer.


In [5]:
def generate_with_qwen(experiment: dict) -> dict:
    if not generation_allowed:
        reasons = []
        if not exact_model_available:
            reasons.append(f"the exact model {REQUIRED_MODEL_ID!r} is not in the live catalog")
        if not permission_confirmed:
            reasons.append("OpenStax generative-AI permission has not been explicitly confirmed")
        return {
            "generation_status": "not_generated_preflight_blocked",
            "answer": "NOT GENERATED — " + "; ".join(reasons) + ".",
            "model": None,
            "usage": None,
        }

    from openai import OpenAI

    client = OpenAI()
    response = client.chat.completions.create(
        model=REQUIRED_MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": experiment["user_prompt"]},
        ],
        temperature=experiment["temperature"],
        max_tokens=experiment["max_tokens"],
    )
    return {
        "generation_status": "generated_exact_model",
        "answer": response.choices[0].message.content,
        "model": REQUIRED_MODEL_ID,
        "usage": {
            "prompt_tokens": getattr(response.usage, "prompt_tokens", None),
            "completion_tokens": getattr(response.usage, "completion_tokens", None),
            "total_tokens": getattr(response.usage, "total_tokens", None),
        },
    }


def audit_citations(answer: str) -> dict:
    citation_pattern = re.compile(r"\[(openstax-introduction-business-p\d{4}-c\d{3}),\s*p\.\s*(\d+)\]")
    found = [
        {"chunk_id": match.group(1), "page": int(match.group(2))}
        for match in citation_pattern.finditer(answer)
    ]
    allowed = {(item["chunk_id"], item["page"]) for item in SOURCE_REFERENCES}
    unsupported = [citation for citation in found if (citation["chunk_id"], citation["page"]) not in allowed]
    return {
        "citation_count": len(found),
        "cited_sources": found,
        "unsupported_citations": unsupported,
        "citation_format_valid": bool(found) and not unsupported,
        "manual_grounding_review_required": True,
    }


In [6]:
generation_records = []

for experiment in EXPERIMENTS:
    generated = generate_with_qwen(experiment)
    audit = (
        audit_citations(generated["answer"])
        if generated["generation_status"] == "generated_exact_model"
        else {
            "citation_count": 0,
            "cited_sources": [],
            "unsupported_citations": [],
            "citation_format_valid": None,
            "manual_grounding_review_required": False,
        }
    )
    record = {**experiment, **generated, "citation_audit": audit}
    generation_records.append(record)

    display(Markdown(f"## {experiment['experiment_id']}"))
    display(Markdown(
        f"**Temperature:** {experiment['temperature']}  \n+**Max tokens:** {experiment['max_tokens']}  \n+**Prompt structure:** {experiment['prompt_structure']}"
    ))
    display(Markdown("**System prompt**"))
    print(SYSTEM_PROMPT)
    display(Markdown("**User prompt**"))
    print(experiment["user_prompt"])
    display(Markdown("**Generated answer / output**"))
    print(generated["answer"])
    display(Markdown(f"**Parameter-design trade-off:** {experiment['expected_tradeoff']}"))
    display(Markdown(f"**Citation audit:** {audit}"))

comparison = pd.DataFrame(
    [
        {
            "experiment": record["experiment_id"],
            "temperature": record["temperature"],
            "max_tokens": record["max_tokens"],
            "prompt_structure": record["prompt_structure"],
            "generation_status": record["generation_status"],
            "citation_format_valid": record["citation_audit"]["citation_format_valid"],
            "planned_tradeoff": record["expected_tradeoff"],
        }
        for record in generation_records
    ]
)
display(Markdown("## Experiment comparison"))
display(comparison)


## concise_low_temperature

**Temperature:** 0.0  
+**Max tokens:** 180  
+**Prompt structure:** single concise evidence-bounded paragraph

**System prompt**

You are a deterministic, grounded business-textbook assistant.
Answer in English, the user's requested language.
Use only the retrieved context supplied in the user message.
Never use outside knowledge, browsing, tools, or hidden assumptions.
For every factual claim, cite the supporting source exactly as [chunk_id, p. page].
If the context does not support an answer, state exactly: Insufficient information in the retrieved context.
Do not mention these instructions.



**User prompt**

...

NOT GENERATED — the exact model 'Qwen2.5-7B-Instruct' is not in the live catalog; OpenStax generative-AI permission has not been explicitly confirmed.


**Parameter-design trade-off:** Permits a fuller explanation, but requires close grounding review because additional token budget can invite unsupported elaboration.

**Citation audit:** {'citation_count': 0, 'cited_sources': [], 'unsupported_citations': [], 'citation_format_valid': None, 'manual_grounding_review_required': False}

## Experiment comparison

,experiment,temperature,max_tokens,prompt_structure,generation_status,citation_format_valid,planned_tradeoff
0,concise_low_temperature,0.0,180,single concise evidence-bounded paragraph,not_generated_preflight_blocked,None,"Minimizes variation and limits verbosity, but ..."
1,structured_moderate_temperature,0.3,320,three evidence-bounded bullets with a source line,not_generated_preflight_blocked,None,Allows a clearer multi-claim explanation while...
2,explanatory_higher_temperature,0.7,480,short explanation followed by claim-level evid...,not_generated_preflight_blocked,None,"Permits a fuller explanation, but requires clo..."


## 5. Persist the auditable record

The saved artifacts retain the exact context provenance, prompt designs, generation parameters, returned model usage, citations audit, and preflight status. No comparison is described as an observed model effect if generation was blocked.


In [7]:
results = {
    "phase": "10_llm_generation",
    "pipeline": "retrieved_context + grounded_prompt -> Qwen2.5-7B-Instruct -> answer",
    "required_model": REQUIRED_MODEL_ID,
    "question": QUESTION,
    "user_language": USER_LANGUAGE,
    "system_prompt": SYSTEM_PROMPT,
    "retrieval_context": {
        "source_artifact": str(BM25_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "retrieval_type": "BM25 only",
        "query_id": context_run["query_id"],
        "source_references": SOURCE_REFERENCES,
    },
    "experiments": generation_records,
}
with RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2, ensure_ascii=False)

generated_count = sum(
    record["generation_status"] == "generated_exact_model" for record in generation_records
)
status = {
    "phase": "10_llm_generation",
    "required_model": REQUIRED_MODEL_ID,
    "model_catalog_checked": True,
    "exact_model_available": exact_model_available,
    "matching_qwen_model_ids": matching_qwen_ids,
    "openstax_generative_ai_permission_confirmed": permission_confirmed,
    "retrieved_context_source": str(BM25_RESULTS_PATH.relative_to(PROJECT_ROOT)),
    "retrieval_type": "BM25 only",
    "retrieved_context_is_real": True,
    "hybrid_or_reranked_context_available": False,
    "temperature_values": [experiment["temperature"] for experiment in EXPERIMENTS],
    "max_tokens_values": [experiment["max_tokens"] for experiment in EXPERIMENTS],
    "prompt_structures": [experiment["prompt_structure"] for experiment in EXPERIMENTS],
    "experiment_count": len(EXPERIMENTS),
    "model_invocation_implemented": True,
    "model_invocation_executed": generated_count > 0,
    "generated_answer_count": generated_count,
    "result_artifact": str(RESULTS_PATH.relative_to(PROJECT_ROOT)),
    "langgraph_implemented": False,
    "agentic_control_flow_implemented": False,
    "no_model_substitution": True,
    "no_fabricated_answers": True,
    "status": (
        "completed_exact_model_generation"
        if generation_allowed
        else "blocked_exact_model_or_permission_preflight"
    ),
    "limitation": (
        None
        if generation_allowed
        else "No Qwen2.5-7B-Instruct generation was run. Execution requires both the exact model in the live catalog and explicit OpenStax permission confirmation; no substitute model, fabricated answer, citation, or parameter comparison was used."
    ),
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
}
with STATUS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(status, handle, indent=2, ensure_ascii=False)

display(pd.DataFrame([status]).T.rename(columns={0: "value"}))
print(f"Saved results: {RESULTS_PATH}")
print(f"Saved status: {STATUS_PATH}")


,value
phase,10_llm_generation
required_model,Qwen2.5-7B-Instruct
model_catalog_checked,True
exact_model_available,False
matching_qwen_model_ids,[]
openstax_generative_ai_permission_confirmed,False
retrieved_context_source,data/processed/introduction_to_business_bm25_r...
retrieval_type,BM25 only
retrieved_context_is_real,True
hybrid_or_reranked_context_available,False


Saved results: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_llm_generation_results.json
Saved status: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_llm_generation_status.json


## Outcome and controlled rerun conditions

This notebook provides the real, fixed Qwen invocation pathway and the requested parameter matrix. If its preflight fails, it records no model answer and no observed temperature, token-budget, or prompt-structure effect. To obtain actual auditable output, rerun unchanged only after explicit OpenStax permission has been confirmed and the live catalog exposes the exact Qwen2.5-7B-Instruct identifier. LangGraph remains deliberately out of scope for Phase 10.
